# Baseline Comparison: Transformer (Base) from Scratch
## PyTorch Implementation of Sequence-to-Sequence Transformer Baseline

**Associated Research Paper:**
*"Developing Neural Machine Translation Tool: Kokborok–English, a Step Toward Preserving Indigenous Languages"*
- **Authors:** Sanchali Das, Pranjal, Mainak Sarkar (CRC Press / Taylor & Francis, 2026)

---
### Context & Architectural Baseline (Table 2 in Paper)
In low-resource machine translation research, standard sequence-to-sequence baselines trained purely from scratch struggle due to limited parallel data:

| Model | BLEU | Comment |
| :--- | :--- | :--- |
| **Seq2Seq (RNN + Attention)** | 5 – 8 | Weak long-range modeling on small data |
| **Transformer (Base from Scratch)** | 9 – 13 | Requires massive corpora to generalize |
| **M2M100 (Multilingual Fine-Tuned)** | **70.56** | Leverages cross-lingual transfer learning |

This notebook provides the implementation of the Transformer Base from scratch architecture in PyTorch.

### 1. Model Architecture: Positional Encoding and Transformer Module

In [ ]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 512,
        nhead: int = 8,
        num_encoder_layers: int = 6,
        num_decoder_layers: int = 6,
        dim_feedforward: int = 2048,
        dropout: float = 0.1,
        pad_idx: int = 0
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None):
        src_emb = self.pos_encoder(self.src_embedding(src))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt))
        out = self.transformer(
            src_emb,
            tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.fc_out(out)

print("Seq2SeqTransformer architecture defined.")

### 2. Autoregressive Beam Search Translation Function

In [ ]:
def translate_sentence(sentence, model, kok_sp, en_sp, max_len=50, beam_size=5, device="cuda"):
    model.eval()
    token_ids = kok_sp.encode(sentence, out_type=int)
    src_tensor = torch.tensor([token_ids], dtype=torch.long, device=device)
    
    # Beam search initialization
    bos_id = en_sp.bos_id() if en_sp.bos_id() != -1 else 2
    eos_id = en_sp.eos_id() if en_sp.eos_id() != -1 else 3
    
    beams = [(torch.tensor([[bos_id]], dtype=torch.long, device=device), 0.0)]
    
    with torch.no_grad():
        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[0, -1].item() == eos_id:
                    candidates.append((seq, score))
                    continue
                out = model(src_tensor, seq)
                log_probs = torch.log_softmax(out[:, -1, :], dim=-1)
                topk_probs, topk_indices = torch.topk(log_probs, beam_size)
                for i in range(beam_size):
                    cand_seq = torch.cat([seq, topk_indices[:, i:i+1]], dim=-1)
                    cand_score = score + topk_probs[0, i].item()
                    candidates.append((cand_seq, cand_score))
            beams = sorted(candidates, key=lambda x: x[1] / len(x[0][0]), reverse=True)[:beam_size]
            if all(seq[0, -1].item() == eos_id for seq, _ in beams):
                break
                
    best_seq = beams[0][0][0].tolist()
    return en_sp.decode([idx for idx in best_seq if idx not in [bos_id, eos_id]])